# IRS EO BMF data research for Climate Org Mapping

This notebook profiles the supplied IRS Exempt Organizations Business Master File (EO BMF) data and evaluates what it can—and cannot—support in the Climate Org Mapping project. It analyzes both the complete nationwide extract and the NTEE subset currently selected for the prototype.

## Questions investigated

1. What files, columns, and record volume are present?
2. Are schemas, EINs, state partitions, and addresses internally consistent?
3. How complete are individual columns?
4. What do IRS status, subsection, affiliation, organization, filing, and NTEE fields mean?
5. How representative is the project's selected environmental subset?
6. How usable are addresses and financial fields?
7. What duplicate signals exist?
8. Which project schema fields are directly supported, weakly inferred, or unsupported?
9. What classification defects are visible in the generated map data?

The analysis uses only the Python standard library plus IPython's built-in display helpers. No pandas, plotting package, or network call is required.

## Executive findings from the supplied files

The most recent full scan produced these results:

- **51 state/DC CSV files**, 324.48 MiB, with a consistent 28-column schema.
- **1,960,014 records and 1,960,014 unique EINs**; no blank or duplicate EIN was observed across these files.
- Every row had street, city, state, and ZIP text, and every record's `STATE` matched its state file. This is mailing-address completeness—not proof of a physical or operating location.
- **375,988 records (19.2%) use a PO Box**, which limits street geocoding.
- `NTEE_CD` is blank for **573,099 records (29.2%)**.
- The current **17-code** environment allowlist selects **20,577 records (1.05%)**. Sixteen configured codes occur in the files; configured code `C00` has no rows.
- Within the selected subset, **19,678 records (95.6%) are subsection 03**, normally meaning section 501(c)(3).
- The selected subset contains 36 repeated-name groups covering 131 rows and 330 repeated mailing-address groups covering 757 rows. These are review signals, not automatic duplicates.
- Half of selected records report zero assets, income, and revenue in the latest BMF amount fields. Zero can reflect the filing population and filing method; it should not be interpreted as organizational inactivity without Form 990 context.
- IRS EO BMF is strong for identity, tax status, mailing address, and broad exempt-purpose classification. It does **not** directly provide websites, mission summaries, audiences, climate strategy, program activity, operational reach, or GCC verification.

The reproducible cells below regenerate and expand these findings.

## Official IRS interpretation

The analysis follows these official sources:

- [EO BMF download page](https://www.irs.gov/charities-non-profits/exempt-organizations-business-master-file-extract-eo-bmf)
- [EO BMF Information Sheet and code definitions](https://www.irs.gov/pub/irs-soi/eo-info.pdf)
- [Publication 5926: EOBMF Dataset Guide](https://www.irs.gov/pub/irs-pdf/p5926.pdf)
- [IRS EO/BMF processing manual](https://www.irs.gov/irm/part25/irm_25-007-001)

Important interpretation constraints from the IRS documentation:

- EO BMF is a **cumulative**, monthly updated extract containing the most recent BMF information.
- State is based on the filing/headquarters address and may not represent where the organization operates.
- Publication 5926 says the downloadable extract includes EO status codes 01, 02, and 25. The supplied files also contain 6,643 status-12 records, so the observed data and the one-page guide are not fully aligned; this notebook reports rather than silently discards them.
- Self-declared organizations and churches or other organizations that never applied for recognition may be absent.
- `NTEE_CD` describes an organization's **primary exempt activity** at a broad taxonomy level.
- Legacy `ACTIVITY` codes became obsolete after adoption of NTEE in 1995; older organizations may still contain them.
- Financial amounts come from the most recent Form 990-series information available in the BMF and are not a longitudinal financial dataset.
- The dataset contains mailing addresses, not coordinates and not guaranteed physical service locations.

In [ ]:
from __future__ import annotations

import csv
import json
import math
import re
import time
from collections import Counter
from pathlib import Path
try:
    from IPython.display import HTML, Markdown, display
except ImportError:
    # Keep the analysis executable in minimal Python environments. A real
    # notebook frontend supplies richer HTML/Markdown display objects.
    HTML = Markdown = lambda value: value
    display = print

# Support launching Jupyter from either the repository root or experimentation/.
ROOT = Path.cwd().resolve()
if not (ROOT / 'Data').exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / 'Data' / 'IRS EO BMF'
CODES_PATH = ROOT / 'Docs' / 'Research and Requirements' / 'NTEE_Codes_to_Consider.txt'
MAP_DATA_PATH = ROOT / 'app' / 'data' / 'organizations.geojson'
CACHE_PATH = ROOT / 'experimentation' / 'irs_eda_cache.json'
ANALYSIS_VERSION = 2

CSV_FILES = sorted(DATA_DIR.glob('eo_*.csv'))
SELECTED_CODES = {
    line.strip().upper()
    for line in CODES_PATH.read_text(encoding='utf-8').splitlines()
    if line.strip() and not line.lstrip().startswith('#')
}

print(f'Repository: {ROOT}')
print(f'IRS files: {len(CSV_FILES)}')
print(f'Raw size: {sum(path.stat().st_size for path in CSV_FILES) / 1024**2:,.2f} MiB')
print(f'Selected NTEE codes: {len(SELECTED_CODES)}')

In [ ]:
FIELD_DESCRIPTIONS = {
    'EIN': 'Nine-digit IRS employer identification number',
    'NAME': 'Primary legal organization name',
    'ICO': 'In-care-of person for correspondence',
    'STREET': 'Mailing street address',
    'CITY': 'Mailing city',
    'STATE': 'Mailing/headquarters state',
    'ZIP': 'Mailing ZIP code',
    'GROUP': 'Group exemption number',
    'SUBSECTION': 'Internal Revenue Code exemption subsection',
    'AFFILIATION': 'Independent, parent, intermediate, or subordinate relationship',
    'CLASSIFICATION': 'One to four IRS classification codes',
    'RULING': 'Exemption ruling date in YYYYMM form',
    'DEDUCTIBILITY': 'Whether contributions are deductible',
    'FOUNDATION': 'Public charity/private foundation classification',
    'ACTIVITY': 'Legacy activity codes; obsolete since NTEE adoption in 1995',
    'ORGANIZATION': 'Legal organization form such as corporation or association',
    'STATUS': 'IRS exempt-organization status code',
    'TAX_PERIOD': 'Most recent tax period in YYYYMM form',
    'ASSET_CD': 'Binned asset-size code',
    'INCOME_CD': 'Binned income-size code',
    'FILING_REQ_CD': 'Primary Form 990 filing requirement',
    'PF_FILING_REQ_CD': 'Form 990-PF filing requirement',
    'ACCT_PD': 'Fiscal-year ending month',
    'ASSET_AMT': 'Latest reported end-of-year assets',
    'INCOME_AMT': 'IRS-computed latest income amount',
    'REVENUE_AMT': 'Latest Form 990/990-EZ revenue amount',
    'NTEE_CD': 'Primary National Taxonomy of Exempt Entities code',
    'SORT_NAME': 'Secondary, trade, chapter, or local name',
}

STATUS_LABELS = {
    '01': 'Unconditional exemption',
    '02': 'Conditional exemption (legacy)',
    '12': 'Formal exemption not granted; active EO-return filer (see IRS IRM)',
    '25': 'Terminating private-foundation status under 507(b)(1)(B)',
}
ORGANIZATION_LABELS = {
    '1': 'Corporation', '2': 'Trust', '3': 'Co-operative',
    '4': 'Partnership', '5': 'Association',
    '6': 'Observed value not defined in the current information sheet',
    '0': 'Observed value not defined in the current information sheet',
}
AFFILIATION_LABELS = {
    '1': 'Central, no group exemption', '2': 'Intermediate, no group exemption',
    '3': 'Independent', '6': 'Parent under group ruling',
    '7': 'Intermediate under group ruling', '8': 'Church/501(c)(1) parent',
    '9': 'Subordinate under group ruling', '0': 'Unspecified',
}
SUBSECTION_LABELS = {
    '03': '501(c)(3): charitable, religious, educational, scientific, etc.',
    '04': '501(c)(4): civic leagues and social-welfare organizations',
    '05': '501(c)(5): labor, agricultural, and horticultural organizations',
    '06': '501(c)(6): business leagues, chambers, and trade associations',
    '07': '501(c)(7): social and recreational clubs',
    '08': '501(c)(8): fraternal beneficiary societies',
    '12': '501(c)(12): benevolent life insurance and mutual utility organizations',
    '13': '501(c)(13): cemetery companies',
    '19': '501(c)(19): veterans organizations',
    '25': '501(c)(25): title-holding corporations or trusts',
}

display(HTML('<table><tr><th>Column</th><th>Meaning</th></tr>' + ''.join(
    f'<tr><td><code>{field}</code></td><td>{description}</td></tr>'
    for field, description in FIELD_DESCRIPTIONS.items()
) + '</table>'))

## Streaming profiler

The next cell scans every CSV once. It retains only counters, identifier sets, and small selected-subset summaries instead of loading 1.96 million dictionaries into memory. Aggregate results are cached using file sizes, modification times, selected-code configuration, and an analysis version. Change `FORCE_RESCAN` to `True` after altering analysis logic or source files.

In [ ]:
def manifest() -> dict:
    return {
        'analysis_version': ANALYSIS_VERSION,
        'selected_codes': sorted(SELECTED_CODES),
        'files': [
            {'name': path.name, 'size': path.stat().st_size, 'mtime_ns': path.stat().st_mtime_ns}
            for path in CSV_FILES
        ],
    }

def percentile(values: list[int], proportion: float) -> float | int | None:
    if not values:
        return None
    ordered = sorted(values)
    position = (len(ordered) - 1) * proportion
    lower, upper = math.floor(position), math.ceil(position)
    if lower == upper:
        return ordered[lower]
    return round(ordered[lower] * (upper - position) + ordered[upper] * (position - lower), 2)

def counter_dict(counter: Counter) -> dict[str, int]:
    return dict(counter.most_common())

def scan_irs_files() -> dict:
    started = time.time()
    expected_headers = None
    schema_mismatches = []
    total_rows = 0
    blank_eins = duplicate_eins = state_file_mismatches = 0
    complete_addresses = po_boxes = 0
    unique_eins = set()
    missing = Counter()
    selected_missing = Counter()
    categorical_fields = [
        'STATE', 'STATUS', 'SUBSECTION', 'AFFILIATION', 'ORGANIZATION',
        'FOUNDATION', 'FILING_REQ_CD', 'NTEE_CD', 'TAX_PERIOD',
    ]
    counts = {field: Counter() for field in categorical_fields}
    selected_counts = {field: Counter() for field in categorical_fields}
    selected_names = Counter()
    selected_addresses = Counter()
    financial_fields = ['ASSET_AMT', 'INCOME_AMT', 'REVENUE_AMT']
    financial = {field: Counter() for field in financial_fields}
    financial_totals = {field: {'sum': 0, 'min': None, 'max': None} for field in financial_fields}
    selected_financial = {field: [] for field in financial_fields}
    selected_rows = 0

    for path in CSV_FILES:
        expected_state = path.stem[-2:].upper()
        with path.open('r', encoding='latin-1', newline='') as stream:
            reader = csv.DictReader(stream)
            if expected_headers is None:
                expected_headers = reader.fieldnames
            elif reader.fieldnames != expected_headers:
                schema_mismatches.append(path.name)

            for row in reader:
                total_rows += 1
                for field, value in row.items():
                    if not (value or '').strip():
                        missing[field] += 1

                ein = (row.get('EIN') or '').strip()
                if not ein:
                    blank_eins += 1
                elif ein in unique_eins:
                    duplicate_eins += 1
                else:
                    unique_eins.add(ein)

                if (row.get('STATE') or '').strip().upper() != expected_state:
                    state_file_mismatches += 1
                if all((row.get(field) or '').strip() for field in ('STREET', 'CITY', 'STATE', 'ZIP')):
                    complete_addresses += 1
                street = (row.get('STREET') or '').strip().upper()
                if re.search(r'\bP\.?\s*O\.?\s+BOX\b', street):
                    po_boxes += 1

                for field in categorical_fields:
                    counts[field][(row.get(field) or '').strip() or '<blank>'] += 1
                for field in financial_fields:
                    raw = (row.get(field) or '').strip()
                    if not raw:
                        financial[field]['missing'] += 1
                        continue
                    try:
                        value = int(raw)
                    except ValueError:
                        financial[field]['invalid'] += 1
                        continue
                    bucket = 'zero' if value == 0 else 'positive' if value > 0 else 'negative'
                    financial[field][bucket] += 1
                    totals = financial_totals[field]
                    totals['sum'] += value
                    totals['min'] = value if totals['min'] is None else min(totals['min'], value)
                    totals['max'] = value if totals['max'] is None else max(totals['max'], value)

                ntee_code = (row.get('NTEE_CD') or '').strip().upper()
                if ntee_code not in SELECTED_CODES:
                    continue
                selected_rows += 1
                for field, value in row.items():
                    if not (value or '').strip():
                        selected_missing[field] += 1
                for field in categorical_fields:
                    selected_counts[field][(row.get(field) or '').strip() or '<blank>'] += 1
                selected_names[(row.get('NAME') or '').strip()] += 1
                selected_addresses[(
                    (row.get('STREET') or '').strip(), (row.get('CITY') or '').strip(),
                    (row.get('STATE') or '').strip(), (row.get('ZIP') or '').strip(),
                )] += 1
                for field in financial_fields:
                    raw = (row.get(field) or '').strip()
                    if raw:
                        try:
                            selected_financial[field].append(int(raw))
                        except ValueError:
                            pass

    duplicate_name_counts = [count for name, count in selected_names.items() if name and count > 1]
    duplicate_address_counts = [count for address, count in selected_addresses.items() if address[0] and count > 1]
    return {
        'manifest': manifest(),
        'elapsed_seconds': round(time.time() - started, 2),
        'file_count': len(CSV_FILES),
        'size_bytes': sum(path.stat().st_size for path in CSV_FILES),
        'columns': expected_headers,
        'schema_mismatch_files': schema_mismatches,
        'rows': total_rows,
        'unique_eins': len(unique_eins),
        'blank_eins': blank_eins,
        'duplicate_ein_rows': duplicate_eins,
        'state_file_mismatches': state_file_mismatches,
        'complete_addresses': complete_addresses,
        'po_boxes': po_boxes,
        'missing': counter_dict(missing),
        'counts': {field: counter_dict(value) for field, value in counts.items()},
        'financial': {
            field: {**counter_dict(financial[field]), **financial_totals[field]}
            for field in financial_fields
        },
        'selected': {
            'rows': selected_rows,
            'missing': counter_dict(selected_missing),
            'counts': {field: counter_dict(value) for field, value in selected_counts.items()},
            'duplicate_name_groups': len(duplicate_name_counts),
            'duplicate_name_rows': sum(duplicate_name_counts),
            'duplicate_address_groups': len(duplicate_address_counts),
            'duplicate_address_rows': sum(duplicate_address_counts),
            'financial_percentiles': {
                field: {
                    str(proportion): percentile(values, proportion)
                    for proportion in (0, .25, .5, .75, .9, .99, 1)
                }
                for field, values in selected_financial.items()
            },
        },
    }

In [ ]:
FORCE_RESCAN = False
cached = None
if CACHE_PATH.exists() and not FORCE_RESCAN:
    candidate = json.loads(CACHE_PATH.read_text(encoding='utf-8'))
    if candidate.get('manifest') == manifest():
        cached = candidate

if cached is None:
    print('Scanning all IRS files; this takes roughly 30 seconds on the current machine...')
    profile = scan_irs_files()
    CACHE_PATH.write_text(json.dumps(profile, indent=2) + '\n', encoding='utf-8')
    print(f"Scan completed in {profile['elapsed_seconds']:.2f} seconds and cached at {CACHE_PATH}")
else:
    profile = cached
    print(f'Loaded current aggregate cache from {CACHE_PATH}')

In [ ]:
def percent(part: int | float, whole: int | float) -> str:
    return '—' if not whole else f'{part / whole:.1%}'

def rows_table(rows, headers):
    head = ''.join(f'<th>{header}</th>' for header in headers)
    body = ''.join('<tr>' + ''.join(f'<td>{cell}</td>' for cell in row) + '</tr>' for row in rows)
    display(HTML(f'<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>'))

selected = profile['selected']
overview = [
    ('Files', f"{profile['file_count']:,}"),
    ('Raw size', f"{profile['size_bytes'] / 1024**2:,.2f} MiB"),
    ('Columns', f"{len(profile['columns']):,}"),
    ('All records', f"{profile['rows']:,}"),
    ('Unique EINs', f"{profile['unique_eins']:,}"),
    ('Duplicate EIN rows', f"{profile['duplicate_ein_rows']:,}"),
    ('Selected environment records', f"{selected['rows']:,} ({percent(selected['rows'], profile['rows'])})"),
]
rows_table(overview, ('Measure', 'Value'))

## Completeness and structural integrity

A blank is not necessarily an error. For example, `SORT_NAME` and `ICO` are optional, financial values depend on filing requirements and available returns, and NTEE is absent for a material share of records. Completeness must be interpreted according to the field's legal meaning.

In [ ]:
completeness_rows = []
for field in profile['columns']:
    missing_all = profile['missing'].get(field, 0)
    missing_selected = selected['missing'].get(field, 0)
    completeness_rows.append((
        f'<code>{field}</code>',
        FIELD_DESCRIPTIONS[field],
        f'{missing_all:,}', percent(missing_all, profile['rows']),
        f'{missing_selected:,}', percent(missing_selected, selected['rows']),
    ))
rows_table(completeness_rows, ('Field', 'Meaning', 'All blank', 'All blank %', 'Selected blank', 'Selected blank %'))

assert not profile['schema_mismatch_files'], profile['schema_mismatch_files']
assert profile['blank_eins'] == 0
assert profile['duplicate_ein_rows'] == 0
assert profile['state_file_mismatches'] == 0
print('Integrity checks passed: consistent headers, populated unique EINs, and correct state partitions.')

## Geographic distribution

These counts describe mailing/headquarters addresses—not program reach. Raw counts should not be interpreted as per-capita climate-organization density without population denominators and a validated definition of climate organization.

In [ ]:
all_states = profile['counts']['STATE']
selected_states = selected['counts']['STATE']
state_rows = [
    (state, f'{count:,}', f"{selected_states.get(state, 0):,}", percent(selected_states.get(state, 0), count))
    for state, count in sorted(all_states.items(), key=lambda item: item[1], reverse=True)
]
rows_table(state_rows, ('State', 'All EO BMF records', 'Selected records', 'Selected share'))

## Tax status and legal structure

`ORGANIZATION` is the IRS legal-form field. It is not equivalent to the project's desired Organization Type taxonomy: for example, an IRS corporation could be an academic institution, municipality-affiliated entity, nonprofit, or coalition. Likewise, `SUBSECTION` is a tax-law category and does not describe climate strategy.

In [ ]:
def decoded_rows(counter_dict, labels, total):
    return [
        (code, labels.get(code, 'See IRS information sheet'), f'{count:,}', percent(count, total))
        for code, count in counter_dict.items()
    ]

display(Markdown('### EO status — all records'))
rows_table(decoded_rows(profile['counts']['STATUS'], STATUS_LABELS, profile['rows']), ('Code', 'Meaning', 'Records', 'Share'))
display(Markdown('### Subsection — selected environment records'))
rows_table(decoded_rows(selected['counts']['SUBSECTION'], SUBSECTION_LABELS, selected['rows']), ('Code', 'Meaning', 'Records', 'Share'))
display(Markdown('### Legal organization form — all records'))
rows_table(decoded_rows(profile['counts']['ORGANIZATION'], ORGANIZATION_LABELS, profile['rows']), ('Code', 'Meaning', 'Records', 'Share'))
display(Markdown('### Affiliation — all records'))
rows_table(decoded_rows(profile['counts']['AFFILIATION'], AFFILIATION_LABELS, profile['rows']), ('Code', 'Meaning', 'Records', 'Share'))

## NTEE coverage and current project selection

The selected codes are all in NTEE's `C` Environment major group, but they span materially different purposes: advocacy, professional associations, research, fundraising, pollution control, recycling, conservation, energy, beautification, environmental education, and not-elsewhere-classified activity. Membership in this set identifies an environmental-taxonomy candidate; it does not prove active climate work.

In [ ]:
ntee_rows = [
    (code, f'{selected["counts"]["NTEE_CD"].get(code, 0):,}', percent(selected['counts']['NTEE_CD'].get(code, 0), selected['rows']))
    for code in sorted(SELECTED_CODES)
]
rows_table(ntee_rows, ('Selected NTEE code', 'Records', 'Share of selected subset'))

blank_ntee = profile['counts']['NTEE_CD'].get('<blank>', 0)
print(f'NTEE is blank for {blank_ntee:,} of {profile["rows"]:,} records ({percent(blank_ntee, profile["rows"])}).')

## Address and geocoding suitability

The address columns are structurally complete in the supplied files, but three distinctions matter:

1. Mailing address is not necessarily a physical office or program location.
2. Headquarters state is not service reach.
3. PO Boxes cannot generally produce an organization street point.

Consequently, Census-geocoded results should be labeled as address-range estimates, PO Box/nonmatches need a fallback or review path, and no IRS address should be presented as organization-verified.

In [ ]:
address_rows = [
    ('Complete street/city/state/ZIP text', f"{profile['complete_addresses']:,}", percent(profile['complete_addresses'], profile['rows'])),
    ('PO Box mailing addresses', f"{profile['po_boxes']:,}", percent(profile['po_boxes'], profile['rows'])),
    ('State/file mismatches', f"{profile['state_file_mismatches']:,}", percent(profile['state_file_mismatches'], profile['rows'])),
    ('Repeated selected-subset address groups', f"{selected['duplicate_address_groups']:,}", 'Review signal'),
    ('Rows in repeated selected-subset address groups', f"{selected['duplicate_address_rows']:,}", percent(selected['duplicate_address_rows'], selected['rows'])),
]
rows_table(address_rows, ('Measure', 'Count', 'Interpretation/share'))

## Financial fields

These values are useful as latest-return context but require care. They are not synchronized to one common fiscal year, missingness depends on filing requirements and available returns, and extreme values make averages misleading. The selected-subset percentile table therefore emphasizes medians and upper percentiles.

In [ ]:
financial_rows = []
for field, stats in profile['financial'].items():
    present = profile['rows'] - stats.get('missing', 0)
    financial_rows.append((
        field, f"{present:,}", percent(present, profile['rows']),
        f"{stats.get('zero', 0):,}", f"{stats.get('positive', 0):,}",
        f"{stats.get('negative', 0):,}", f"${stats.get('max', 0):,}",
    ))
rows_table(financial_rows, ('Field', 'Present', 'Present %', 'Zero', 'Positive', 'Negative', 'Maximum'))

percentile_rows = []
for field, values in selected['financial_percentiles'].items():
    percentile_rows.append((field, *[f"${values[str(p)]:,.0f}" for p in (0, .25, .5, .75, .9, .99, 1)]))
rows_table(percentile_rows, ('Field', 'Min', '25th', 'Median', '75th', '90th', '99th', 'Max'))

## Identity and duplicate signals

EIN is unique across the supplied state files and is the strongest available source identifier. Repeated names do not necessarily indicate duplicates: chapters, affiliates, similarly named trusts, and distinct legal entities can have different EINs. Repeated addresses can indicate shared offices, fiscal sponsors, registered agents, parent/subordinate relationships, or actual duplication. Future source merging should use exact EIN first and treat name/address similarity as review evidence rather than automatic identity.

In [ ]:
identity_rows = [
    ('Unique EINs', f"{profile['unique_eins']:,}", percent(profile['unique_eins'], profile['rows'])),
    ('Blank EINs', f"{profile['blank_eins']:,}", percent(profile['blank_eins'], profile['rows'])),
    ('Duplicate EIN rows', f"{profile['duplicate_ein_rows']:,}", percent(profile['duplicate_ein_rows'], profile['rows'])),
    ('Repeated selected-name groups', f"{selected['duplicate_name_groups']:,}", 'Manual/entity-resolution review'),
    ('Rows in repeated selected-name groups', f"{selected['duplicate_name_rows']:,}", percent(selected['duplicate_name_rows'], selected['rows'])),
    ('Repeated selected-address groups', f"{selected['duplicate_address_groups']:,}", 'Manual/entity-resolution review'),
    ('Rows in repeated selected-address groups', f"{selected['duplicate_address_rows']:,}", percent(selected['duplicate_address_rows'], selected['rows'])),
]
rows_table(identity_rows, ('Measure', 'Count', 'Meaning/share'))

## Audit of current generated classifications

This section inspects `app/data/organizations.geojson` when present. It does not treat its derived GCC fields as IRS facts. Instead, it measures how deterministic rules transformed the IRS evidence and looks for known string-matching collisions.

In [ ]:
if MAP_DATA_PATH.exists():
    map_payload = json.loads(MAP_DATA_PATH.read_text(encoding='utf-8'))
    map_rows = [feature.get('properties', {}) for feature in map_payload.get('features', [])]
    derived_fields = [
        'gcc_organization_type', 'gcc_audience_focus', 'gcc_resilience',
        'gcc_activity', 'gcc_action_category', 'gcc_climate_category',
        'gcc_reach', 'gcc_verification',
    ]
    distribution_rows = []
    for field in derived_fields:
        values = Counter(row.get(field, '<missing>') for row in map_rows)
        distribution_rows.append((f'<code>{field}</code>', '<br>'.join(
            f'{value}: {count:,} ({percent(count, len(map_rows))})'
            for value, count in values.most_common()
        )))
    rows_table(distribution_rows, ('Derived field', 'Distribution'))

    transportation = [row for row in map_rows if row.get('gcc_climate_category') == 'Transportation']
    recycle_collision = [
        row for row in transportation
        if any(term in row.get('name', '').casefold() for term in ('recycling', 'recycle'))
    ]
    farmington_collision = [
        row for row in map_rows
        if row.get('gcc_climate_category') == 'Food & Food Waste'
        and 'farmington' in row.get('name', '').casefold()
    ]
    print(f'Map records: {len(map_rows):,}')
    print(f'Transportation records: {len(transportation):,}')
    print(f'Transportation records containing recycling/recycle: {len(recycle_collision):,}')
    print(f'Food-category records containing Farmington: {len(farmington_collision):,}')
    display(Markdown('**Interpretation:** unrestricted substring rules cause `cycling` inside `recycling` to produce Transportation, and `farm` inside place names such as Farmington to produce Food & Food Waste. These are classifier defects, not IRS data defects.'))
else:
    print('Generated map data is absent; skip derived-classification audit.')

## Suitability for the Climate Org Mapping schema

| Target information | IRS support | Recommended treatment |
|---|---|---|
| Legal identity / EIN | Strong | Use EIN as the IRS source key; retain source provenance |
| Legal name | Strong | Preserve `NAME`; use `SORT_NAME` as an alternate name when present |
| Mailing address | Strong as mailing data | Geocode with precision labels; do not claim physical location |
| Exempt status | Strong within EOBMF scope | Decode `STATUS`; do not confuse IRS exemption with GCC verification |
| Tax subsection / legal form | Strong | Decode `SUBSECTION` and `ORGANIZATION`; do not equate them with product taxonomy |
| Broad primary exempt activity | Moderate | Preserve NTEE as a source classification and candidate-selection signal |
| Current activity or program details | Weak/absent | Obtain official-site or Form 990 evidence |
| Website and summary | Absent | Research separately and retain evidence URLs |
| Audience focus | Absent | Use evidence-based enrichment; allow multiple audiences if required |
| Adaptation / mitigation | Absent | Use evidence-based classification or `Unknown` |
| Advocacy / action | Weak | NTEE may suggest broad purpose but cannot prove actual balance |
| Climate category | Weak to moderate | NTEE can nominate candidates; validate using organization evidence |
| Geographic reach | Absent | Never derive service reach from mailing state alone |
| Verification | Absent | Only the GCC review/claim workflow can set Verified |
| Latest financial scale | Moderate with caveats | Show filing period and missingness; avoid treating zero as inactivity |

## Conclusions

1. **Use EO BMF as a discovery and identity foundation, not a finished climate directory.** It reliably supplies EIN, legal name, tax attributes, mailing address, and a broad NTEE signal.
2. **The 20,577-record allowlist is a candidate pool.** Selected C codes cover broad environmental purposes and do not independently establish climate relevance.
3. **Separate source facts from product claims.** IRS fields should remain immutable source observations; evidence-based or human-reviewed values belong in separate fields with provenance.
4. **Prefer `Unknown` to unsupported certainty.** Audience, reach, resilience, and activity should not be filled merely to make filters look complete.
5. **Repair deterministic string rules before using them even as provisional signals.** Use token/phrase boundaries, collision tests, field-specific confidence, and no cascade from one weak guess into several fields.
6. **Use EIN for exact IRS identity, then perform explicit entity resolution for future sources.** Shared names and addresses require review-aware matching.
7. **Geocode cautiously.** About one in five nationwide records uses a PO Box, and all IRS addresses represent mailing/headquarters information rather than verified program locations.
8. **Do not use BMF financial zeros as an activity test.** Filing requirements and return availability must be considered, ideally with Form 990 data.
9. **Treat data-dictionary mismatches as explicit quality findings.** Status 12 and organization codes 0/6 occur locally even though the compact current guides do not fully describe them; downstream validation must preserve and flag such values.
10. **Validate enrichment on a labeled pilot.** Website discovery, Jev classification, confidence thresholds, and multi-value schema choices should be evaluated before nationwide publication.